# P1.01 · GBM monotônico, calibrado e registrado
**Motor de concessão de crédito E2E** · camada **L3**

[![Abrir no Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/1thirteeng3/ds-ai-cookbook/blob/colab/pt/projects/P1-credit/P1.01-e2e-credit-decision-engine/notebooks/31-gbm-monotonic-calibrated.ipynb)

| Nível | Runtime Colab | Tempo (smoke / full) | Documento |
|---|---|---|---|
| 🟠 L3 | cpu | 4 / 30 min | [index.pt.md](../index.pt.md) |

> LightGBM com restrições monotônicas de negócio, ajuste de hiperparâmetros com Optuna, calibração escolhida sem tocar no teste, códigos de motivo por cliente e registro no MLflow.

## 0. Parâmetros

`RUN_MODE="smoke"` usa 10% da base (ou o gêmeo sintético com `OFFLINE=True`) e roda em minutos. `RUN_MODE="full"` usa a base inteira e produz os resultados oficiais. `OFFLINE=True` só existe para testes e CI: nunca tire conclusões dele.

In [ ]:
# Parameters (papermill overrides this cell)
RUN_MODE = "smoke"   # "smoke" | "full"
SEED = 42
DATA_DIR = "data"    # Colab + Drive: "/content/drive/MyDrive/cookbook-data"
OFFLINE = False      # True: synthetic twin (tests/CI). Never for full results.
SAMPLE_FRAC = 0.1    # smoke only
N_BOOT = 200         # bootstrap resamples for metric uncertainty
N_TRIALS = 40   # Optuna trials in full mode (smoke uses 8)

## 1. Setup

No Colab, instala o pacote `cookbook` do repositório privado (segredo `GH_TOKEN`) e as versões exatas usadas nos resultados. NumPy, pandas e scikit-learn ficam com as versões do Colab e são registrados no arquivo de resultados. Se a instalação atualizar algum deles, use *Ambiente de execução > Reiniciar sessão* e rode a partir da seção 0. Para baixar o Home Credit, crie o segredo `KAGGLE_API_TOKEN` (ou `KAGGLE_USERNAME` e `KAGGLE_KEY`) e aceite as regras da competição no kaggle.com com a mesma conta.

In [ ]:
import sys

IN_COLAB = "google.colab" in sys.modules
PACKAGES = {
    "optbinning": "1.0.0",
    "lightgbm": "4.7.0",
    "optuna": "5.0.0",
    "mlflow": "3.18.0",
    "fastapi": "0.162.0",
    "httpx": "0.28.1",
    "polars": "2.1.0",
    "pyarrow": "26.0.1",
    "kaggle": "2.2.5",
    "joblib": "1.6.0",
    "matplotlib": "3.11.2",
}

if IN_COLAB and not sys.modules.get("cookbook"):
    from google.colab import userdata

    token = userdata.get("GH_TOKEN")
    pins = " ".join(f"{k}=={v}" for k, v in PACKAGES.items())
    %pip install -q uv
    !uv pip install --system -q {pins} "cookbook[credit,kaggle] @ git+https://{token}@github.com/1thirteeng3/cookbokk.git@main"

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

from cookbook import credit as cr
from cookbook import home_credit as hc
from cookbook.data import sample
from cookbook.env import colab_secrets_to_env, hardware_info, quiet_logs, set_seeds, versions
from cookbook.results import Timer, save_result

quiet_logs()  # before heavy imports: silences optional-solver probing
import json

import joblib
import lightgbm as lgb
import mlflow
import optuna
from mlflow import MlflowClient
from sklearn.metrics import brier_score_loss

set_seeds(SEED)
colab_secrets_to_env()
if RUN_MODE == "full" and OFFLINE:
    raise ValueError("RUN_MODE='full' needs real data (OFFLINE=False).")

SOURCE = "home-credit-synthetic" if OFFLINE else "home-credit"
ARTIFACTS = Path(DATA_DIR) / "artifacts" / "P1.01" / RUN_MODE
ARTIFACTS.mkdir(parents=True, exist_ok=True)
HW = hardware_info()
VERSIONS = versions([*PACKAGES, "numpy", "pandas", "scikit-learn"])
pd.set_option("display.float_format", "{:.4f}".format)
pd.set_option("display.width", 140)
print(SOURCE, RUN_MODE, ARTIFACTS, HW)

In [ ]:
missing = [f for f in ["features.parquet", "feature_meta.json", "l1_scores.parquet"] if not (ARTIFACTS / f).exists()]
if missing:
    raise FileNotFoundError(
        f"Missing {missing} in {ARTIFACTS}. Run notebooks 10 and 30 first "
        f"with the same RUN_MODE and DATA_DIR. / Rode 10 e 30 antes."
    )

## 2. Dados

Lemos a tabela de features e o vocabulário congelado do notebook 30. `train` ajusta o modelo, `valid` decide parada antecipada, hiperparâmetros e calibração, e `test` só é tocado para medir.

In [ ]:
feats = pd.read_parquet(ARTIFACTS / "features.parquet")
meta = json.loads((ARTIFACTS / "feature_meta.json").read_text())
DATASET = meta["dataset"]
COLS = meta["model_columns"]
X, _ = hc.to_categorical(feats[COLS], meta["categories"])
y = feats[hc.TARGET].to_numpy()
part = feats["SPLIT"].to_numpy()
tr, va, te = (part == s for s in ("train", "valid", "test"))
print({s: int(m.sum()) for s, m in zip(("train", "valid", "test"), (tr, va, te))})

## 3. Solução

### 3.1 Restrições monotônicas

Algumas relações têm direção conhecida e precisam ser respeitadas para o modelo ser defensável perante comitê e regulador: score externo maior nunca pode aumentar o risco, mais atraso no bureau nunca pode reduzi-lo. O LightGBM impõe isso por feature (`-1` = risco cai quando a feature sobe).

In [ ]:
MONOTONE = {
    "EXT_SOURCE_1": -1, "EXT_SOURCE_2": -1, "EXT_SOURCE_3": -1, "EXT_SOURCE_MEAN": -1,
    "EXT_SOURCE_MIN": -1, "EXT_SOURCE_PROD": -1, "ANNUITY_INCOME": 1,
    "BUREAU_MAX_DAYS_OVERDUE": 1, "BUREAU_OVERDUE_COUNT": 1, "BUREAU_BAD_DEBT_COUNT": 1,
}
constraints = [MONOTONE.get(c, 0) for c in COLS]
BASE = {"objective": "binary", "metric": "auc", "verbose": -1, "seed": SEED, "bagging_freq": 1}


def make_datasets(eval_mask):
    # A fresh Dataset per training run: reusing one built with monotone constraints for an
    # unconstrained model crashes LightGBM 4.x natively (segfault), see Pitfalls.
    dtr = lgb.Dataset(X[tr], y[tr], params={"feature_pre_filter": False})
    return dtr, lgb.Dataset(X[eval_mask], y[eval_mask], reference=dtr)


def train_booster(params, monotone=True, eval_mask=None):
    eval_mask = va if eval_mask is None else eval_mask
    mono = {"monotone_constraints": constraints, "monotone_constraints_method": "advanced"}
    dtr, dev = make_datasets(eval_mask)
    return lgb.train(BASE | params | (mono if monotone else {}), dtr, 5000, valid_sets=[dev],
                     callbacks=[lgb.early_stopping(150, verbose=False)])

### 3.2 Busca de hiperparâmetros (Optuna, TPE)

Cada tentativa treina com parada antecipada na validação e devolve o AUC de validação. O espaço é pequeno e focado nos parâmetros que mais importam em GBMs tabulares.

In [ ]:
def objective(trial):
    params = {
        "learning_rate": trial.suggest_float("learning_rate", 0.02, 0.1, log=True),
        "num_leaves": trial.suggest_int("num_leaves", 8, 128, log=True),
        "min_child_samples": trial.suggest_int("min_child_samples", 20, 500, log=True),
        "feature_fraction": trial.suggest_float("feature_fraction", 0.4, 1.0),
        "bagging_fraction": trial.suggest_float("bagging_fraction", 0.5, 1.0),
        "lambda_l2": trial.suggest_float("lambda_l2", 1e-3, 10.0, log=True),
        "cat_smooth": trial.suggest_float("cat_smooth", 1.0, 50.0, log=True),
    }
    booster = train_booster(params)
    trial.set_user_attr("best_iteration", booster.best_iteration)
    return booster.best_score["valid_0"]["auc"]

optuna.logging.set_verbosity(optuna.logging.WARNING)
n_trials = N_TRIALS if RUN_MODE == "full" else 8
study = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=SEED))
with Timer() as t_tune:
    study.optimize(objective, n_trials=n_trials)
BEST = study.best_params
print(f"{n_trials} trials in {t_tune.seconds:.0f}s · best valid AUC {study.best_value:.4f}")
BEST

In [ ]:
with Timer() as t_fit:
    booster = train_booster(BEST, monotone=True)
booster_free = train_booster(BEST, monotone=False)
with Timer() as t_pred:
    raw_test = booster.predict(X[te], num_iteration=booster.best_iteration)
raw_valid = booster.predict(X[va], num_iteration=booster.best_iteration)
raw_test_free = booster_free.predict(X[te], num_iteration=booster_free.best_iteration)
print("best iteration:", booster.best_iteration)

### 3.3 Calibração sem tocar no teste

O PD vai entrar numa conta de lucro (notebook 32), então precisa ser **calibrado**, não só ordenar bem. Comparamos identidade, Platt e isotônica com validação cruzada em duas metades da validação, escolhemos pelo Brier e reajustamos o vencedor na validação inteira. O teste continua intocado.

In [ ]:
rng = np.random.default_rng(SEED)
half = rng.random(va.sum()) < 0.5
y_valid = y[va]
cal_scores = {}
for method in ("identity", "platt", "isotonic"):
    briers = []
    for fit_mask in (half, ~half):
        cal = cr.ProbabilityCalibrator(method).fit(raw_valid[fit_mask], y_valid[fit_mask])
        briers.append(brier_score_loss(y_valid[~fit_mask], cal.predict(raw_valid[~fit_mask])))
    cal_scores[method] = float(np.mean(briers))
CAL_METHOD = min(cal_scores, key=cal_scores.get)
calibrator = cr.ProbabilityCalibrator(CAL_METHOD).fit(raw_valid, y_valid)
pd_test = calibrator.predict(raw_test)
print("chosen:", CAL_METHOD)
pd.Series(cal_scores, name="cross-fitted Brier on valid")

## 4. Validação

Gini, KS, Brier e ECE no teste, comparados com o scorecard L1 e com o modelo sem restrições (quanto a monotonicidade custa).

In [ ]:
y_test = y[te]
l1 = pd.read_parquet(ARTIFACTS / "l1_scores.parquet").set_index(hc.ID)
pd_l1 = l1.loc[feats.loc[te, hc.ID], "pd_l1"].to_numpy()
METRICS = {
    "gini": cr.bootstrap(cr.gini, y_test, pd_test, n=N_BOOT, seed=SEED),
    "ks": cr.bootstrap(cr.ks, y_test, pd_test, n=N_BOOT, seed=SEED),
    "brier": cr.bootstrap(brier_score_loss, y_test, pd_test, n=N_BOOT, seed=SEED),
    "ece": {"mean": round(cr.ece(y_test, pd_test), 6), "std": None, "n": 1},
    "best_iteration": {"mean": booster.best_iteration, "std": None, "n": 1},
}
comparison = pd.DataFrame({
    "L1 scorecard": {"gini": cr.gini(y_test, pd_l1), "ece": cr.ece(y_test, pd_l1)},
    "L3 GBM unconstrained (raw)": {"gini": cr.gini(y_test, raw_test_free), "ece": cr.ece(y_test, raw_test_free)},
    "L3 GBM monotonic (raw)": {"gini": cr.gini(y_test, raw_test), "ece": cr.ece(y_test, raw_test)},
    "L3 GBM monotonic + calibration": {"gini": cr.gini(y_test, pd_test), "ece": cr.ece(y_test, pd_test)},
}).T
comparison

### 4.1 Códigos de motivo

Para cada cliente, as três features que mais empurraram o PD para cima (contribuições SHAP exatas de árvore, `pred_contrib`). É o que sustenta a explicação de uma recusa.

In [ ]:
sample_idx = np.argsort(-pd_test)[:3].tolist() + np.argsort(pd_test)[:2].tolist()
contrib = booster.predict(X[te].iloc[sample_idx], pred_contrib=True, num_iteration=booster.best_iteration)[:, :-1]
pd.DataFrame({
    "SK_ID_CURR": feats.loc[te, hc.ID].to_numpy()[sample_idx],
    "pd": pd_test[sample_idx],
    "reasons": [", ".join(r) for r in cr.reason_codes(contrib, COLS)],
})

### 4.2 Registro no MLflow

Parâmetros, métricas, o modelo e o calibrador vão para o MLflow (backend SQLite local; em produção, um servidor compartilhado). O modelo é registrado e recebe o alias `champion`, que o deploy usa em vez de um número de versão.

In [ ]:
MODEL_NAME = "p1-01-credit-pd"
mlflow.set_tracking_uri(f"sqlite:///{(ARTIFACTS / 'mlflow.db').resolve()}")
client = MlflowClient()
if client.get_experiment_by_name("P1.01") is None:   # artifacts next to the DB, not in the cwd
    client.create_experiment("P1.01", artifact_location=(ARTIFACTS / "mlruns").resolve().as_uri())
mlflow.set_experiment("P1.01")
joblib.dump(calibrator, ARTIFACTS / "calibrator.joblib")
with mlflow.start_run(run_name=f"lgbm-monotonic-{RUN_MODE}") as run:
    mlflow.log_params({**BEST, "calibration": CAL_METHOD, "n_trials": n_trials, "run_mode": RUN_MODE})
    mlflow.log_metrics({f"test_{k}": v["mean"] for k, v in METRICS.items()})
    info = mlflow.lightgbm.log_model(booster, name="model", registered_model_name=MODEL_NAME)
    mlflow.log_artifact(str(ARTIFACTS / "calibrator.joblib"))
version = info.registered_model_version
client.set_registered_model_alias(MODEL_NAME, "champion", version)
print(f"registered {MODEL_NAME} v{version} as 'champion' (run {run.info.run_id[:8]})")

## 5. Eixos de variação

### 5.1 Restrições monotônicas: custo em Gini

Já medido na seção 4: compare as linhas *unconstrained* e *monotonic*. Diferença pequena = a restrição sai quase de graça e compra defensabilidade.

### 5.2 Número de tentativas do Optuna

Melhor AUC de validação acumulado por tentativa: mostra quando a busca para de compensar.

In [ ]:
history = study.trials_dataframe()[["number", "value"]]
history["best_so_far"] = history["value"].cummax()
history.iloc[[0, len(history) // 4, len(history) // 2, -1]]

## 6. Armadilhas demonstradas

### 6.1 Parada antecipada no teste

**Sintoma:** Gini de teste melhor que o real. **Causa:** o teste escolheu o número de árvores. **Correção:** parar na validação; o teste só mede.

### 6.2 Calibrar com as previsões de treino

**Sintoma:** PDs subestimados no teste (ECE alto). **Causa:** no treino, o modelo é confiante demais. **Correção:** calibrar em dados que o modelo não viu.

In [ ]:
cheat = train_booster(BEST, monotone=True, eval_mask=te)
raw_train = booster.predict(X[tr], num_iteration=booster.best_iteration)
bad_cal = cr.ProbabilityCalibrator(CAL_METHOD if CAL_METHOD != "identity" else "isotonic").fit(raw_train, y[tr])
pd.DataFrame({
    "test gini": {"early stop on valid (correct)": cr.gini(y_test, raw_test),
                  "early stop on test (leak)": cr.gini(y_test, cheat.predict(X[te], num_iteration=cheat.best_iteration))},
    "test ECE": {"calibrated on valid (correct)": cr.ece(y_test, pd_test),
                 "calibrated on train (wrong)": cr.ece(y_test, bad_cal.predict(raw_test))},
})

## 7. Resultados

Grava as métricas e os artefatos que as próximas etapas consomem: modelo (`lgbm.txt`), calibrador, cartão do modelo e PDs de validação e teste.

In [ ]:
booster.save_model(str(ARTIFACTS / "lgbm.txt"), num_iteration=booster.best_iteration)
card = {"model_name": MODEL_NAME, "model_version": str(version), "calibration": CAL_METHOD,
        "best_params": BEST, "best_iteration": booster.best_iteration, "monotone": MONOTONE,
        "run_mode": RUN_MODE, "dataset": DATASET["name"]}
(ARTIFACTS / "model_card.json").write_text(json.dumps(card, indent=2))

scored = feats.loc[va | te, [hc.ID, "SPLIT", hc.TARGET, "AMT_CREDIT"]].copy()
scored["raw_l3"] = booster.predict(X[va | te], num_iteration=booster.best_iteration)
scored["pd_l3"] = calibrator.predict(scored["raw_l3"])
scored.to_parquet(ARTIFACTS / "l3_predictions.parquet", index=False)

common = dict(recipe_id="P1.01", notebook="31-gbm-monotonic-calibrated", run_mode=RUN_MODE,
              hardware=HW, versions=VERSIONS, dataset=DATASET)
save_result(**common, stack="lightgbm-monotonic-calibrated",
            params={"seed": SEED, "n_trials": n_trials, "calibration": CAL_METHOD, **BEST},
            metrics=METRICS,
            time_s={"train": round(t_tune.seconds + t_fit.seconds, 3),
                    "inference_per_1k": round(t_pred.seconds / te.sum() * 1000, 6)})
save_result(**common, stack="lightgbm-unconstrained", variant="no monotonic constraints",
            params={"seed": SEED, **BEST},
            metrics={"gini": cr.bootstrap(cr.gini, y_test, raw_test_free, n=N_BOOT, seed=SEED)},
            time_s={"train": t_fit.seconds, "inference_per_1k": None})
sorted(p.name for p in ARTIFACTS.iterdir())

## 8. Conclusão

- Um PD calibrado, monotônico onde o negócio exige, versionado e explicável por cliente.
- A comparação com a L1 diz quanto a complexidade rende; a linha sem restrição diz quanto a defensabilidade custa.
- **Próximo passo:** [32 · política de decisão](32-decision-policy.ipynb) transforma PD em aprovar ou recusar.